In [1]:
#--import libraries
import numpy as np
import math

In [9]:
#--define q,k,v
seq_length=4
d_q=8
d_k=8
d_v=8

q=np.random.randn(seq_length, d_q)
k=np.random.randn(seq_length, d_k)
v=np.random.randn(seq_length, d_v)

print(f'q vector :{q} \n with var :{q.var()} \n and mean :{q.mean()}')
print(f'k vector :{k} \n with var :{k.var()} \n and mean :{k.mean()}')
print(f'v vector :{v} \n with var :{v.var()} \n and mean :{v.mean()}')

q vector :[[-0.09022246  0.77778127 -0.57131605  1.19369835 -0.69419356 -0.0719925
  -0.19689366 -1.49030392]
 [ 1.28035968  0.50696877  0.47442329  1.46499558  0.49380727  2.26701556
  -1.16358568  0.35465539]
 [-0.4391743   0.40625305 -0.92079159  0.26356441 -1.25360963  1.46856952
   0.21361533 -1.23419892]
 [-0.71981488 -0.05311896 -1.13175752  0.97710778 -0.86148704 -0.09283852
   1.09463714  0.07925116]] 
 with var :0.8548800097698313 
 and mean :0.07285638606647803
k vector :[[ 0.33769624 -3.76220657 -1.07685199 -0.3530212   0.31483448 -0.0801607
   0.2049416  -0.64068232]
 [ 1.35905413  1.92115114  0.25404018 -0.35014633 -1.12286468 -0.12110335
   0.52486335  0.3006377 ]
 [ 0.87547755  0.31266356  1.64558239  1.59735676  1.44114915 -0.92259963
  -0.22042134 -0.06721027]
 [ 1.17227088 -0.31003791  0.45503863  1.09902275  1.90171186 -0.58504194
  -0.45113096 -2.06928313]] 
 with var :1.3596770672229814 
 and mean :0.11202281339703529
v vector :[[-0.66529552  0.43504551  0.6494903

In [14]:
#---dot product
# k,k.T
dot_product=np.matmul(q,k.T)
print(f'dot_product :{dot_product} with shape :{dot_product.shape}') #---> (seq_len,seq_len)

dot_product :[[-2.06114502  1.045332    0.3403577   2.59967202]
 [-2.9949533   0.98847885  3.25300487  2.57352032]
 [-0.45609599  0.82826512 -4.47737345 -1.55575811]
 [ 0.74033935 -0.13302229 -2.35088707 -2.51028234]] with shape :(4, 4)


In [15]:
#--why scaling
not_scaled=dot_product
scaled=dot_product/math.sqrt(d_k)

print(f'without scaling dot_product var :{not_scaled.var()}')
print(f'with scaling dot_product var :{scaled.var()}')

without scaling dot_product var :4.627203439551396
with scaling dot_product var :0.5784004299439245


In [ ]:
#--create mask
mask=np.ones((seq_length,seq_length))
print(f'mask :{mask}')
mask=np.tril(mask)
print(f'mask :{mask}')
mask[mask==0]=float('-inf')
print(f'mask :{mask}')
mask[mask==1]=0
print(f'mask :{mask}')


mask :[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]
mask :[[1. 0. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 1. 0.]
 [1. 1. 1. 1.]]
mask :[[  1. -inf -inf -inf]
 [  1.   1. -inf -inf]
 [  1.   1.   1. -inf]
 [  1.   1.   1.   1.]]
mask :[[  0. -inf -inf -inf]
 [  0.   0. -inf -inf]
 [  0.   0.   0. -inf]
 [  0.   0.   0.   0.]]


In [20]:
#--apply softmax on with and without mask
def softmax(x):
    return np.exp(x)/np.sum(np.exp(x),axis=-1)

In [24]:
print(f'scaled :{scaled}')
print(f'scaled + mask  :{scaled+mask}')


scaled :[[-0.72872481  0.36958067  0.12033462  0.91912286]
 [-1.05887589  0.34948005  1.1501109   0.90987683]
 [-0.16125428  0.29283594 -1.58299056 -0.55004355]
 [ 0.26174949 -0.04703048 -0.83116409 -0.88751883]]
scaled + mask  :[[-0.72872481        -inf        -inf        -inf]
 [-1.05887589  0.34948005        -inf        -inf]
 [-0.16125428  0.29283594 -1.58299056        -inf]
 [ 0.26174949 -0.04703048 -0.83116409 -0.88751883]]


In [22]:
softmax_without_mask=softmax(scaled)
softmax_with_mask=softmax(scaled+mask)
print(f'softmax_without_mask :{softmax_without_mask}')
print(f'softmax_with_mask :{softmax_with_mask}')

softmax_without_mask :[[0.08671288 0.19535359 0.37929801 0.80861433]
 [0.06233053 0.19146607 1.06220103 0.80117232]
 [0.15294418 0.18092209 0.06906147 0.18607609]
 [0.23347541 0.12879217 0.14647042 0.13277824]]
softmax_with_mask :[[1.         0.         0.         0.        ]
 [0.71881513 0.8035065  0.         0.        ]
 [1.76379992 0.75925764 0.08568598 0.        ]
 [2.69251123 0.54048925 0.18172884 0.13277824]]


In [30]:
#--create a single function
def scaled_dot_product(q,k,v,mask=None):
    d_k=q.shape[1]
    print(f'd_k :{d_k}')
    
    dot_product=np.matmul(q,k.T)
    scaled_dot_product=dot_product/math.sqrt(d_k)
    print(f'dot_product var :{dot_product.var()}')
    print(f'scaled_dot_product var :{scaled_dot_product.var()}')

    #---apply masking
    if mask is not None:
        scaled_dot_product=scaled_dot_product+mask

    #--apply softmax
    new_attention=softmax(scaled_dot_product)

    #--get new values
    new_values=np.matmul(new_attention,v)

    return new_attention, new_values

In [31]:
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [33]:
# new_attention, new_values=scaled_dot_product(q,k,v,mask=None)
new_attention, new_values=scaled_dot_product(q,k,v,mask=mask)

print(f'new_attention :{new_attention} with shape :{new_attention.shape}') #--> (seq_len, seq_len)
print(f'new_values :{new_values} with shape :{new_values.shape}') #--> (seq_len, v)


d_k :8
dot_product var :4.627203439551396
scaled_dot_product var :0.5784004299439245
new_attention :[[1.         0.         0.         0.        ]
 [0.71881513 0.8035065  0.         0.        ]
 [1.76379992 0.75925764 0.08568598 0.        ]
 [2.69251123 0.54048925 0.18172884 0.13277824]] with shape :(4, 4)
new_values :[[-0.66529552  0.43504551  0.64949037  0.29133573 -1.06464733 -2.90675803
  -0.05436122  0.42142433]
 [-0.25673136  1.24116297  0.63423368 -0.35617836  0.25727483 -1.8335432
  -1.17895491  0.49026077]
 [-1.15052379  1.76150642  1.301195   -0.12230246 -0.90175229 -4.85737664
  -1.2168143   0.99582341]
 [-1.83223048  2.24970623  1.79638763  0.10801161 -2.21267797 -7.8217456
  -0.75765263  1.47440846]] with shape :(4, 8)
